---세션 다시 시작 및 모든 셀 실행---

노트북을 처음부터 다시 실행하려면 Colab 메뉴에서 **런타임(Runtime) > 모두 실행(Run all)**을 선택하세요.


In [35]:
import pandas as pd

file_path = '/content/한국인터넷진흥원_침해사고 공격 IoC 지표_20260211.csv'
df = pd.read_csv(file_path, encoding='utf-8') # Trying 'utf-8' encoding

### 유사한 공격 유형 의미론적 그룹화

'수행 행위' 컬럼의 범주를 단순화하고 해석 가능성을 높이기 위해 의미론적으로 유사한 공격 유형을 그룹화하겠습니다. 예를 들어, 'SQL Injection', '원격 접근', '무차별 대입 공격', '악성 파일 접근', '계정 탈취', '피싱'과 관련된 다양한 구문들을 더 일반적인 단일 범주로 통합할 수 있습니다. 이는 빈도에 따른 필터링을 하기 전에 더 넓은 공격 패턴을 이해하는 데 도움이 됩니다.

In [57]:
import pandas as pd

# 유사한 행동을 공통 카테고리로 매핑하기 위한 딕셔너리 업데이트
# '무작위 대입 공격'과 '불법 광고 게시글 업로드' 등을 추가로 매핑
attack_type_mapping = {
    'SQL Injection': 'SQL Injection',
    'SQL Injection 시도': 'SQL Injection',
    'SQL Injection 공격': 'SQL Injection',
    'SQL 인젝션 공격': 'SQL Injection',
    'SQL Injection 공격 수행': 'SQL Injection',
    '블라인드 SQL Injection': 'SQL Injection',

    'RDP 연결': 'Remote Access',
    'RDP 접근': 'Remote Access',
    'RDP 접근 시도': 'Remote Access',
    '원격 접속': 'Remote Access',

    '무차별 대입 공격': 'Brute Force Attack',
    '무차별 대입 공격 시도': 'Brute Force Attack',
    'phpmyadmin 무작위 대입 공격': 'Brute Force Attack',
    '무작위 대입 공격': 'Brute Force Attack', # 통합
    'xmlrpc 무차별 대입 공격': 'Brute Force Attack',

    '악성 파일 접근': 'Malicious File Access',
    '악성 파일 유포지': 'Malicious File Access',
    '악성파일 유포지': 'Malicious File Access',

    '관리자 계정 로그인': 'Account Credential Access',
    '관리자 계정 로그인 및 정보 유출': 'Account Credential Access',
    '관리자 페이지 접근': 'Account Credential Access',
    'phpMyAdmin 로그인 시도': 'Account Credential Access',
    '웹메일 로그인': 'Account Credential Access',

    '피싱 메일 발송지': 'Phishing Related',
    '피싱 메일 발신지': 'Phishing Related',

    'C2 서버': 'C2 Server',
    '웹셸 접근': 'Webshell Access',
    'PHP 정보 노출 페이지 접근': 'PHP Info Disclosure Access',
    '악성코드 유포': 'Malicious Code Distribution',
    '불법 광고 게시글 업로드': 'Other', # 기타로 명시적 매핑
    '기타': 'Other'
}

# 매핑을 적용하여 새로운 그룹화된 컬럼 생성
df['수행 행위_semantic_grouped'] = df['수행 행위'].replace(attack_type_mapping)

print("\n의미론적 그룹화 완료 (Brute Force 통합):")
display(df['수행 행위_semantic_grouped'].value_counts().head(10))


의미론적 그룹화 완료 (Brute Force 통합):


,count
수행 행위_semantic_grouped,
Brute Force Attack,637
SQL Injection,279
Malicious File Access,188
Account Credential Access,106
Remote Access,97
Webshell Access,81
PHP Info Disclosure Access,26
Phishing Related,22
C2 Server,15


### 의미론적 그룹화 후 낮은 빈도의 항목들을 '기타'로 통합

의미론적 그룹화를 거친 후에도 여전히 발생 횟수가 5회 미만인 '수행 행위' 항목들을 '기타' (Other) 카테고리로 통합하여 데이터를 더욱 간결하게 만들겠습니다.

In [54]:
import numpy as np

# --- 1단계: 빈도수가 3 미만인 항목 제거 ---
semantic_action_counts_step1 = df['수행 행위_semantic_grouped'].value_counts()
low_freq_to_remove = semantic_action_counts_step1[semantic_action_counts_step1 < 3].index
df['수행 행위_semantic_grouped'] = df['수행 행위_semantic_grouped'].replace(low_freq_to_remove, np.nan)
df.dropna(subset=['수행 행위_semantic_grouped'], inplace=True)

# --- 2단계: 빈도수가 5 이하인 항목을 '기타'로 통합 ---
semantic_action_counts_step2 = df['수행 행위_semantic_grouped'].value_counts()
low_freq_to_other = semantic_action_counts_step2[semantic_action_counts_step2 <= 5].index
df['수행 행위_semantic_grouped'] = df['수행 행위_semantic_grouped'].replace(low_freq_to_other, '기타')

# 최종 빈도수 갱신
action_counts_after_filter = df['수행 행위_semantic_grouped'].value_counts()
print("\n최종 카테고리 분포 (통합 후):")
display(action_counts_after_filter)


최종 카테고리 분포 (통합 후):


,count
수행 행위_semantic_grouped,
Brute Force Attack,637
SQL Injection,279
Malicious File Access,188
Account Credential Access,106
Remote Access,97
Webshell Access,81
PHP Info Disclosure Access,26
Phishing Related,22
C2 Server,15


### 국가별 가장 빈번한 공격 유형 분석

'공격 IP 국가'와 의미론적으로 그룹화된 '수행 행위'를 기준으로, 각 국가에서 가장 빈번하게 발생하는 공격 유형을 분석해 보겠습니다.

In [59]:
# '공격 IP 국가'와 '수행 행위_semantic_grouped'를 기준으로 그룹화하고 발생 횟수 재계산
attack_by_country_type = df.groupby(['공격 IP 국가', '수행 행위_semantic_grouped']).size().reset_index(name='count')

# 각 국가별로 가장 빈번하게 발생하는 공격 유형 찾기
most_frequent_attack_per_country = attack_by_country_type.loc[attack_by_country_type.groupby('공격 IP 국가')['count'].idxmax()]

print("국가별 가장 빈번하게 발생하는 공격 유형 (통합 후):")
display(most_frequent_attack_per_country.sort_values(by='count', ascending=False).head(10))

국가별 가장 빈번하게 발생하는 공격 유형 (통합 후):


,공격 IP 국가,수행 행위_semantic_grouped,count
192,US,Malicious File Access,118
34,CN,Brute Force Attack,69
154,RU,Brute Force Attack,65
126,NL,Brute Force Attack,60
149,RO,Brute Force Attack,54
121,NG,Account Credential Access,37
44,DE,Brute Force Attack,27
100,JP,SQL Injection,22
83,ID,Brute Force Attack,21
75,HK,Account Credential Access,20


### 1회만 기록된 국가들을 '기타_국가'로 통합

국가별 가장 빈번한 공격 유형 분석 결과에서 'count'가 1인 국가들을 '기타_국가'로 묶어 데이터를 간소화하고 전반적인 패턴을 더 명확하게 파악하겠습니다.

In [60]:
# 'count'가 10 미만인 국가들을 '기타_국가'로 변경 (데이터 최신화)
most_frequent_attack_per_country['공격 IP 국가'] = (
    most_frequent_attack_per_country['공격 IP 국가'].where(
        most_frequent_attack_per_country['count'] >= 10,
        '기타_국가'
    )
)

# '기타_국가'로 변경된 항목들을 다시 그룹화하여 합계 계산
final_grouped_countries = most_frequent_attack_per_country.groupby(
    ['공격 IP 국가', '수행 행위_semantic_grouped']
)['count'].sum().reset_index()

# '기타_국가'를 최종 결과에서 제외
final_grouped_countries = final_grouped_countries[final_grouped_countries['공격 IP 국가'] != '기타_국가']

print("\n대시보드용 국가별 데이터 업데이트 완료:")
display(final_grouped_countries.sort_values(by='count', ascending=False).head(10))


대시보드용 국가별 데이터 업데이트 완료:


,공격 IP 국가,수행 행위_semantic_grouped,count
18,US,Malicious File Access,118
2,CN,Brute Force Attack,69
14,RU,Brute Force Attack,65
12,NL,Brute Force Attack,60
13,RO,Brute Force Attack,54
11,NG,Account Credential Access,37
3,DE,Brute Force Attack,27
10,JP,SQL Injection,22
7,ID,Brute Force Attack,21
5,GB,Brute Force Attack,20


### 상위 15개 '수행 행위' (Action Performed) 항목 선정

현재 DataFrame에서 가장 빈번하게 발생하는 '수행 행위' 상위 15개 항목을 선정하여 주요 공격 유형을 파악하겠습니다.

In [41]:
# '수행 행위_semantic_grouped'의 각 항목별 발생 횟수 계산
action_counts_after_filter = df['수행 행위_semantic_grouped'].value_counts()

# 상위 15개 '수행 행위_semantic_grouped' 항목 선정
top_15_actions = action_counts_after_filter.head(15)

print("필터링 후 상위 15개 의미론적 그룹화된 '수행 행위' 항목:")
display(top_15_actions)

필터링 후 상위 15개 의미론적 그룹화된 '수행 행위' 항목:


,count
수행 행위_semantic_grouped,
Brute Force Attack,637
SQL Injection,279
Malicious File Access,188
Account Credential Access,106
Remote Access,97
Webshell Access,81
PHP Info Disclosure Access,26
Phishing Related,22
C2 Server,15


In [87]:
# @title 📊 사이버 침해사고 분석 대시보드 {display-mode: "form"}
import pandas as pd
import json
from IPython.display import HTML
from google.colab import output

# 1. Data Preparation
top_countries_data = final_grouped_countries.sort_values(by='count', ascending=False).head(10)
top_countries_json = top_countries_data.to_json(orient='records')

all_attacks = action_counts_after_filter.reset_index()
all_attacks.columns = ['attack_type', 'count']

# Ensure '기타' is at the bottom
others_mask = all_attacks['attack_type'] == '기타'
top_attacks_data = pd.concat([all_attacks[~others_mask], all_attacks[others_mask]]).head(15)
top_attacks_json = top_attacks_data.to_json(orient='records')

total_count = int(df.shape[0])
kpi_data = json.dumps({
    "total_incidents": total_count,
    "top_country": str(top_countries_data.iloc[0]['공격 IP 국가']),
    "top_attack": str(top_attacks_data.iloc[0]['attack_type'])
})

# 2. HTML/JS Content
html_template = """
<!DOCTYPE html>
<html>
<head>
    <script src=\"https://cdn.jsdelivr.net/npm/chart.js\"></script>
    <link href=\"https://fonts.googleapis.com/css2?family=Pretendard:wght@400;600;800&display=swap\" rel=\"stylesheet\">
    <style>
        :root {
            --bg-color: #f8fafc;
            --card-bg: #ffffff;
            --text-main: #1e293b;
            --text-muted: #64748b;
            --accent: #3b82f6;
            --border: #e2e8f0;
        }
        body {
            font-family: 'Pretendard', sans-serif;
            background-color: var(--bg-color);
            color: var(--text-main);
            margin: 0;
            padding: 24px;
        }
        .dashboard {
            max-width: 1260px;
            margin: auto;
            display: flex;
            flex-direction: column;
            gap: 24px;
        }
        .kpi-row {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(300px, 1fr));
            gap: 20px;
        }
        .card {
            background: var(--card-bg);
            padding: 24px;
            border-radius: 16px;
            box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.05);
            border: 1px solid var(--border);
            display: flex;
            flex-direction: column;
        }
        .kpi-card { align-items: center; text-align: center; }
        .kpi-label { color: var(--text-muted); font-size: 0.9rem; font-weight: 600; margin-bottom: 8px; }
        .kpi-value { font-size: 2rem; font-weight: 800; color: var(--accent); }
        .chart-row { display: grid; grid-template-columns: 1.1fr 0.9fr; gap: 24px; }
        @media (max-width: 1024px) { .chart-row { grid-template-columns: 1fr; } }
        h2 { margin: 0 0 20px 0; font-size: 1.25rem; font-weight: 700; color: var(--text-main); border-left: 4px solid var(--accent); padding-left: 12px; }
        .canvas-wrapper { position: relative; flex-grow: 1; min-height: 400px; }
    </style>
</head>
<body>
    <div class=\"dashboard\">
        <div class=\"kpi-row\">
            <div class=\"card kpi-card\"><div class=\"kpi-label\">📊 총 침해 사고 건수</div><div class=\"kpi-value\" id=\"kpi-total\">0</div></div>
            <div class=\"card kpi-card\"><div class=\"kpi-label\">🌍 최다 공격 발생국</div><div class=\"kpi-value\" id=\"kpi-top-country\">-</div></div>
            <div class=\"card kpi-card\"><div class=\"kpi-label\">⚔️ 핵심 공격 프로파일</div><div class=\"kpi-value\" id=\"kpi-top-attack\" style=\"font-size:1.4rem\">-</div></div>
        </div>
        <div class=\"chart-row\">
            <div class=\"card\"><h2>📍 국가별 공격 현황 및 핵심 유형 (Top 10)</h2><div class=\"canvas-wrapper\"><canvas id=\"countryChart\"></canvas></div></div>
            <div class=\"card\"><h2>🍕 공격 유형별 비중 (Top 15)</h2><div class=\"canvas-wrapper\"><canvas id=\"attackChart\"></canvas></div></div>
        </div>
    </div>
    <script>
        const kpis = JSON_KPI_DATA;
        const countries = JSON_COUNTRY_DATA;
        const attacks = JSON_ATTACK_DATA;
        const TOTAL_VAL = JSON_TOTAL_COUNT;

        document.getElementById('kpi-total').innerText = kpis.total_incidents.toLocaleString() + '건';
        document.getElementById('kpi-top-country').innerText = kpis.top_country;
        document.getElementById('kpi-top-attack').innerText = kpis.top_attack;

        const vibrantPalette = ['#3b82f6', '#ef4444', '#10b981', '#f59e0b', '#8b5cf6', '#ec4899', '#06b6d4', '#f97316', '#6366f1', '#14b8a6', '#94a3b8', '#475569', '#1e293b', '#fbbf24', '#f472b6'];

        new Chart(document.getElementById('countryChart'), {
            type: 'bar',
            data: {
                labels: countries.map(d => `${d['공격 IP 국가']} [${d['수행 행위_semantic_grouped']}]`),
                datasets: [{
                    label: '탐지 건수',
                    data: countries.map(d => d['count']),
                    backgroundColor: '#3b82f6',
                    borderRadius: 8,
                    barThickness: 25
                }]
            },
            options: {
                indexAxis: 'y',
                responsive: true,
                maintainAspectRatio: false,
                plugins: {
                    legend: { display: false },
                    tooltip: {
                        callbacks: {
                            label: (ctx) => ` 탐지 건수: ${ctx.raw}건`,
                            afterLabel: (ctx) => ` 핵심 공격 유형: ${countries[ctx.dataIndex]['수행 행위_semantic_grouped']}`
                        }
                    }
                },
                scales: {
                    y: { ticks: { font: { size: 11, weight: '600' } } }
                }
            }
        });

        new Chart(document.getElementById('attackChart'), {
            type: 'doughnut',
            data: {
                labels: attacks.map(d => d.attack_type),
                datasets: [{
                    data: attacks.map(d => d.count),
                    backgroundColor: vibrantPalette,
                    hoverOffset: 25,
                    borderWidth: 2,
                    borderColor: '#fff'
                }]
            },
            options: {
                responsive: true, maintainAspectRatio: false,
                plugins: {
                    legend: {
                        position: 'right',
                        labels: { boxWidth: 15, padding: 20, font: { size: 12, family: 'Pretendard', weight: '600' }, usePointStyle: true }
                    },
                    tooltip: {
                        backgroundColor: 'rgba(0,0,0,0.8)',
                        padding: 12,
                        callbacks: {
                            label: (ctx) => {
                                const val = ctx.raw;
                                const pct = ((val / TOTAL_VAL) * 100).toFixed(1);
                                return ` ${ctx.label}: ${val}건 (${pct}%)`;
                            }
                        }
                    }
                }
            }
        });
    </script>
</body>
</html>
"""
final_html = html_template.replace('JSON_KPI_DATA', kpi_data).replace('JSON_COUNTRY_DATA', top_countries_json).replace('JSON_ATTACK_DATA', top_attacks_json).replace('JSON_TOTAL_COUNT', str(total_count))
display(HTML(final_html))